<a href="https://colab.research.google.com/github/fischaaulia/credit-default-prediction/blob/main/2404220004_UTS%20PMML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Setup**

In [2]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

### **Import Data**

In [3]:
df = pd.read_csv("UCI_Credit_Card.csv")

## **Experiment 0 - Data Understanding & Preparation**

In [4]:
print(df.shape) #dimensi
df.info()                # struktur dan tipe data
print(df.isnull().sum().sum())                 # missing value
print(df.drop(columns="ID").duplicated().sum())  # duplicate (tanpa ID)
print(df.describe().T)

TARGET = "default.payment.next.month"
print(df[TARGET].value_counts(normalize=True))   # distribusi target

# Nilai tidak wajar pada kolom kategorikal
for c in ["SEX", "EDUCATION", "MARRIAGE"]:
    print(df[c].value_counts().sort_index())
print(df["PAY_0"].value_counts().sort_index())

(30000, 25)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   ID                          30000 non-null  int64  
 1   LIMIT_BAL                   30000 non-null  float64
 2   SEX                         30000 non-null  int64  
 3   EDUCATION                   30000 non-null  int64  
 4   MARRIAGE                    30000 non-null  int64  
 5   AGE                         30000 non-null  int64  
 6   PAY_0                       30000 non-null  int64  
 7   PAY_2                       30000 non-null  int64  
 8   PAY_3                       30000 non-null  int64  
 9   PAY_4                       30000 non-null  int64  
 10  PAY_5                       30000 non-null  int64  
 11  PAY_6                       30000 non-null  int64  
 12  BILL_AMT1                   30000 non-null  float64
 13  BILL_AMT2          

**Cleaning**

In [5]:
# Cleaning
df = df.drop_duplicates(subset=df.columns.drop("ID")).copy()
df["EDUCATION"] = df["EDUCATION"].replace({0: 4, 5: 4, 6: 4})
df["MARRIAGE"] = df["MARRIAGE"].replace({0: 3})

# Predictor dan target (ID dikeluarkan)
X = df.drop(columns=["ID", TARGET])
y = df[TARGET]

**Train-Test Split**

In [6]:
# Split stratified supaya proporsi default sama di train dan test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [7]:
results = []

def evaluate(name, exp, model):
    cv_f1 = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1").mean()
    model.fit(X_train, y_train)
    p = model.predict(X_test)
    row = {"Experiment": exp, "Model": name, "CV F1": cv_f1,
           "Test Accuracy": accuracy_score(y_test, p),
           "Test Precision": precision_score(y_test, p),
           "Test Recall": recall_score(y_test, p),
           "Test F1": f1_score(y_test, p)}
    results.append(row)
    return row

### **Hasil Pemeriksaan Data**

Dataset terdiri atas 30.000 observasi dan 25 variabel yang seluruhnya bertipe numerik, dengan variabel target default.payment.next.month (0 = tidak default, 1 = default). Pemeriksaan menunjukkan tidak ada missing value, tetapi terdapat 35 baris duplikat apabila kolom ID diabaikan. Distribusi target tidak seimbang, yaitu 77,88% tidak default dan 22,12% default. Pada variabel kategorik ditemukan kode yang tidak terdokumentasi, yakni kode 0, 5, dan 6 pada EDUCATION (total 345 baris) serta kode 0 pada MARRIAGE (54 baris). Variabel PAY_0 sampai PAY_6 bernilai −2 sampai 8, sedangkan BILL_AMT1 sampai BILL_AMT6 memuat nilai negatif yang masih wajar karena dapat menunjukkan kelebihan pembayaran. Rentang LIMIT_BAL, AGE, dan PAY_AMT tergolong masuk akal sehingga tidak ada nilai yang dihapus.

### **Keputusan Preprocessing**

**Variabel ID dikeluarkan.** ID hanya identitas baris dan tidak memiliki makna prediktif, sehingga predictor terdiri dari 23 variabel dan targetnya default.payment.next.month.

**Duplikat dihapus sebelum split.** Setelah ID diabaikan, ditemukan 35 baris identik. Penghapusan dilakukan sebelum pembagian data agar baris yang sama tidak muncul di data latih sekaligus data uji, yang dapat membuat hasil evaluasi terlalu optimistis. Data yang tersisa berjumlah 29.965 baris.

**Kategori tidak terdokumentasi digabung.** Kode 0, 5, dan 6 pada EDUCATION serta kode 0 pada MARRIAGE tidak memiliki definisi resmi. Menghapus baris tersebut akan membuang informasi, sedangkan membiarkannya menambah kategori yang tidak bermakna. Karena itu, nilai tersebut digabung ke kategori "lainnya" yang sudah tersedia pada masing-masing variabel.

**Nilai ekstrem tidak dihapus.** BILL_AMT negatif dan PAY_AMT yang sangat besar merupakan kondisi yang mungkin terjadi pada transaksi kartu kredit, dan bukan kesalahan input. Perbedaan skala antarvariabel akan ditangani melalui scaling pada Experiment 2.

**Pembagian data dilakukan secara stratified.** Data dibagi dengan rasio 80:20 menggunakan stratify=y dan random_state=42, sehingga proporsi kelas default tetap sama pada data latih (23.972 baris) dan data uji (5.993 baris). Pembagian dilakukan sebelum training dan sebelum scaling agar tidak terjadi data leakage.

**Teknik yang tidak diterapkan.** Imputasi tidak dilakukan karena tidak ada missing value. Penghapusan outlier tidak dilakukan karena nilai tersebut masih wajar. Teknik resampling seperti SMOTE juga tidak dipakai pada tahap ini, karena ketidakseimbangan kelas ditangani melalui split stratified dan pemilihan F1 score sebagai metrik utama. Scaling dan encoding sengaja ditunda ke Experiment 2 agar baseline pada Experiment 1 dapat dibandingkan secara adil, dan keduanya dilakukan di dalam Pipeline sehingga hanya mempelajari data latih pada setiap fold.

## **Experiment 1 - Baseline Model**

### **Import library**

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

### **Bangun model baseline**

In [10]:
results = []
lr_base = LogisticRegression(max_iter=1000, random_state=42)

### **5-fold Cross Validation pada training set (metrik: F1)**

In [11]:
cv_f1 = cross_val_score(lr_base, X_train, y_train, cv=cv, scoring="f1").mean()

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

### **Metrik pada test set**

In [13]:
lr_base.fit(X_train, y_train)
y_pred = lr_base.predict(X_test)

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)

print("CV F1-score    :", round(cv_f1, 4))
print("Test Accuracy  :", round(acc, 4))
print("Test Precision :", round(prec, 4))
print("Test Recall    :", round(rec, 4))
print("Test F1-score  :", round(f1, 4))

results.append({"Experiment": 1, "Model": "Baseline (Logistic Regression)",
                "CV F1": cv_f1, "Test Accuracy": acc, "Test Precision": prec,
                "Test Recall": rec, "Test F1": f1})

CV F1-score    : 0.2695
Test Accuracy  : 0.8079
Test Precision : 0.6611
Test Recall    : 0.2707
Test F1-score  : 0.3842


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Model baseline Logistic Regression menghasilkan CV F1 sebesar 0,2695 dan Test F1 sebesar 0,3842. Accuracy tergolong tinggi (0,8079) dan Precision cukup baik (0,6611), tetapi Recall hanya 0,2707, artinya model hanya mendeteksi sekitar 27% nasabah yang benar benar mengalami default. Accuracy yang tinggi dipengaruhi oleh dominasi kelas tidak default sebesar 77,88%, sehingga F1 score dipakai sebagai metrik utama. Nilai CV F1 yang lebih rendah daripada Test F1 menunjukkan bahwa kinerja baseline kurang stabil antar fold, yang dapat disebabkan oleh perbedaan skala fitur yang belum ditangani. Hasil ini menjadi acuan perbandingan untuk eksperimen berikutnya.

## **Experiment 2 - Logistic Regression + Preprocessing**

### **Menentukan kolom kategorikal dan kolom numerik**

In [14]:
cat_cols = ["SEX", "EDUCATION", "MARRIAGE"]
num_cols = [c for c in X_train.columns if c not in cat_cols]

### **Preprocessing: Scaling untuk Numerik, One Hot Encoding untuk Kategorikal**

In [15]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols),
])

#Gabungkan preprocessing dan model dalam satu Pipeline
lr_prep = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(max_iter=1000, random_state=42)),
])

### **5-fold Cross Validation pada training set**

In [16]:
cv_f1 = cross_val_score(lr_prep, X_train, y_train, cv=cv, scoring="f1").mean()

### **Metrik pada test set**

In [17]:
#Latih lalu prediksi test set
lr_prep.fit(X_train, y_train)
y_pred = lr_prep.predict(X_test)

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)

print("CV F1-score    :", round(cv_f1, 4))
print("Test Accuracy  :", round(acc, 4))
print("Test Precision :", round(prec, 4))
print("Test Recall    :", round(rec, 4))
print("Test F1-score  :", round(f1, 4))

results.append({"Experiment": 2, "Model": "Baseline + Preprocessing",
                "CV F1": cv_f1, "Test Accuracy": acc, "Test Precision": prec,
                "Test Recall": rec, "Test F1": f1})

CV F1-score    : 0.3587
Test Accuracy  : 0.8138
Test Precision : 0.7273
Test Recall    : 0.2534
Test F1-score  : 0.3758


### **Bandingkan dengan baseline (Experiment 1)**

In [18]:
pd.DataFrame(results).round(4)

,Experiment,Model,CV F1,Test Accuracy,Test Precision,Test Recall,Test F1
0,1,Baseline (Logistic Regression),0.2695,0.8079,0.6611,0.2707,0.3842
1,2,Baseline + Preprocessing,0.3587,0.8138,0.7273,0.2534,0.3758


### **Apakah preprocessing yang dilakukan memberikan peningkatan performa model?**

Preprocessing pada Experiment 2 memberikan peningkatan performa yang terbatas. Berdasarkan 5 fold Cross Validation, CV F1 meningkat dari 0,2695 menjadi 0,3587, yang menunjukkan bahwa scaling dan encoding membuat model lebih stabil dan lebih baik dalam mengenali kelas default pada data latih. Pada test set, Accuracy naik dari 0,8079 menjadi 0,8138 dan Precision naik dari 0,6611 menjadi 0,7273, tetapi Recall turun dari 0,2707 menjadi 0,2534 sehingga Test F1 sedikit menurun dari 0,3842 menjadi 0,3758. Penurunan Test F1 yang kecil (0,0084) tergolong tidak berarti, sedangkan CV F1 lebih dapat diandalkan karena dihitung dari lima pembagian data, bukan satu pembagian saja. Dengan demikian, preprocessing bermanfaat untuk kestabilan model, namun belum menyelesaikan rendahnya Recall. Hal ini menunjukkan bahwa masalah utama terletak pada ketidakseimbangan kelas, bukan pada skala fitur.

## **Experiment 3 - Model Comparison**

### **Tiga model yang dibandingkan**

In [19]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
}

### **Evaluasi tiap model dengan prosedur yang sama**

In [20]:
for nama, model in models.items():
    pipe = Pipeline([("prep", preprocessor), ("clf", model)])

    # 5-fold CV pada training set
    cv_f1 = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="f1").mean()

    # Latih lalu prediksi test set
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    results.append({"Experiment": 3, "Model": nama,
                    "CV F1": cv_f1,
                    "Test Accuracy": accuracy_score(y_test, y_pred),
                    "Test Precision": precision_score(y_test, y_pred),
                    "Test Recall": recall_score(y_test, y_pred),
                    "Test F1": f1_score(y_test, y_pred)})

### **Hasil**

In [21]:
tabel_exp3 = pd.DataFrame(results)
tabel_exp3 = tabel_exp3[tabel_exp3["Experiment"] == 3].round(4)
tabel_exp3

,Experiment,Model,CV F1,Test Accuracy,Test Precision,Test Recall,Test F1
2,3,Logistic Regression,0.3587,0.8138,0.7273,0.2534,0.3758
3,3,Decision Tree,0.4049,0.7228,0.3799,0.3997,0.3896
4,3,Random Forest,0.4720,0.8119,0.6304,0.3627,0.4605


### **Berdasarkan hasil eksperimen, tentukan model terbaik yang akan digunakan pada Experiment 4.**

Berdasarkan hasil Experiment 3, model terbaik yang dipilih untuk Experiment 4 adalah Random Forest. Pemilihan didasarkan pada F1 score sebagai metrik utama, dengan CV F1 sebagai pertimbangan pertama karena dihitung dari lima pembagian data sehingga lebih dapat diandalkan. Random Forest memperoleh CV F1 tertinggi sebesar 0,4720, lebih tinggi daripada Decision Tree (0,4049) dan Logistic Regression (0,3587). Keunggulan ini dikonfirmasi pada test set, dengan Test F1 sebesar 0,4605 dibandingkan Decision Tree (0,3896) dan Logistic Regression (0,3758). Selisih antara CV F1 dan Test F1 pada Random Forest juga paling kecil (0,0115), yang menunjukkan performanya stabil dan tidak mengalami overfitting yang berarti.

Pada metrik pendukung, Random Forest memiliki Accuracy 0,8119 dan Precision 0,6304 yang cukup baik, dengan Recall 0,3627 yang lebih tinggi daripada Logistic Regression (0,2534). Decision Tree memang memiliki Recall sedikit lebih tinggi (0,3997), tetapi Precision (0,3799) dan Accuracy (0,7228) yang rendah membuat F1-nya tetap di bawah Random Forest. Logistic Regression memiliki Precision tertinggi (0,7273), namun Recall yang sangat rendah membuat F1-nya paling kecil. Dengan demikian, **Random Forest** dipilih karena memberikan keseimbangan terbaik antara Precision dan Recall, bukan karena asumsi bahwa model ensemble pasti lebih baik.

## **Experiment 4 - Hyperparameter Tuning**

### **Pipeline: preprocessing + Random Forest**

In [22]:
rf_pipe = Pipeline([
    ("prep", preprocessor),
    ("clf", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
])

### **Hyperparameter yang diuji**

In [23]:
param_grid = {
    "clf__max_depth": [10, None],               # kedalaman maksimum pohon
    "clf__min_samples_leaf": [1, 5],            # minimum sampel per daun
    "clf__class_weight": [None, "balanced"],    # penanganan ketidakseimbangan kelas
}

### **GridSearchCV dengan 5-fold CV, metrik utama F1**

In [24]:
grid = GridSearchCV(rf_pipe, param_grid, cv=cv, scoring="f1", n_jobs=-1)
grid.fit(X_train, y_train)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('prep',
                                        ColumnTransformer(transformers=[('num',
                                                                         StandardScaler(),
                                                                         ['LIMIT_BAL',
                                                                          'AGE',
                                                                          'PAY_0',
                                                                          'PAY_2',
                                                                          'PAY_3',
                                                                          'PAY_4',
                                                                          'PAY_5',
                                                                          'PAY_6',
                                                                          'BILL_AMT1',
                                                                          'BILL_AMT2',
                                                                          'BILL_AMT3',
                                                                          'BILL_AMT4',
                                                                          'BILL_AMT5',
                                                                          'BILL_AMT6',
                                                                          'PAY_AMT1',
                                                                          'PAY_AMT2',
                                                                          'PAY_AMT3',
                                                                          'PAY_AMT4',
                                                                          'PAY_AMT5',
                                                                          'PAY_AMT6']),
                                                                        ('cat',
                                                                         OneHotEncoder(drop='first',
                                                                                       handle_unknown='ignore'),
                                                                         ['SEX',
                                                                          'EDUCATION',
                                                                          'MARRIAGE'])])),
                                       ('clf',
                                        RandomForestClassifier(n_jobs=-1,
                                                               random_state=42))]),
             n_jobs=-1,
             param_grid={'clf__class_weight': [None, 'balanced'],
                         'clf__max_depth': [10, None],
                         'clf__min_samples_leaf': [1, 5]},
             scoring='f1')

### **Hasil Tuning**

In [25]:
print("Hyperparameter terbaik:", grid.best_params_)
print("Best CV F1-score      :", round(grid.best_score_, 4))

Hyperparameter terbaik: {'clf__class_weight': 'balanced', 'clf__max_depth': 10, 'clf__min_samples_leaf': 5}
Best CV F1-score      : 0.5473


### **Evaluasi model terbaik pada test set**

In [31]:
y_pred = grid.predict(X_test)
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)

print("Test Accuracy :", round(acc, 4))
print("Test Precision:", round(prec, 4))
print("Test Recall   :", round(rec, 4))
print("Test F1-score :", round(f1, 4))

results.append({"Experiment": 4, "Model": "Random Forest (Tuned)",
                "CV F1": grid.best_score_, "Test Accuracy": acc,
                "Test Precision": prec, "Test Recall": rec, "Test F1": f1})

Test Accuracy : 0.7774
Test Precision: 0.4973
Test Recall   : 0.5633
Test F1-score : 0.5283


In [35]:
# Semua kombinasi yang diuji (8 kombinasi), diurutkan dari CV F1 tertinggi
hasil_grid = pd.DataFrame(grid.cv_results_)[
    ["param_clf__max_depth", "param_clf__min_samples_leaf",
     "param_clf__class_weight", "mean_test_score"]
].sort_values("mean_test_score", ascending=False).round(4)
hasil_grid

,param_clf__max_depth,param_clf__min_samples_leaf,param_clf__class_weight,mean_test_score
5,10,5,balanced,0.5473
7,None,5,balanced,0.5409
4,10,1,balanced,0.5405
2,None,1,None,0.4720
0,10,1,None,0.4714
1,10,5,None,0.4710
3,None,5,None,0.4703
6,None,1,balanced,0.4535


### **Perbandingan sebelum vs sesudah tuning**

In [36]:
tabel = pd.DataFrame(results)
tabel[tabel["Model"].isin(["Random Forest", "Random Forest (Tuned)"])].round(4)

,Experiment,Model,CV F1,Test Accuracy,Test Precision,Test Recall,Test F1
4,3,Random Forest,0.4720,0.8119,0.6304,0.3627,0.4605
5,4,Random Forest (Tuned),0.5473,0.7774,0.4973,0.5633,0.5283
6,4,Random Forest (Tuned),0.5473,0.7774,0.4973,0.5633,0.5283
7,4,Random Forest (Tuned),0.5473,0.7774,0.4973,0.5633,0.5283
8,4,Random Forest (Tuned),0.5473,0.7774,0.4973,0.5633,0.5283


### **Tentukan apakah hyperparameter tuning memberikan peningkatan performa.**

Hyperparameter tuning pada Experiment 4 meningkatkan performa model. Hyperparameter yang diuji adalah max_depth (10 dan None), min_samples_leaf (1 dan 5), serta class_weight (None dan balanced), dengan kombinasi terbaik class_weight=balanced, max_depth=10, dan min_samples_leaf=5. Best CV F1 meningkat dari 0,4720 menjadi 0,5473 dan Test F1 meningkat dari 0,4605 menjadi 0,5283. Peningkatan terbesar terjadi pada Recall, yang naik dari 0,3627 menjadi 0,5633, sehingga model lebih banyak mendeteksi nasabah default. Sebagai konsekuensinya, Precision turun dari 0,6304 menjadi 0,4973 dan Accuracy turun dari 0,8119 menjadi 0,7774. Penurunan ini wajar karena model menjadi lebih sering memprediksi kelas default, dan F1 score yang menyeimbangkan Precision dan Recall tetap meningkat. Dengan demikian, tuning terbukti bermanfaat, terutama berkat penanganan ketidakseimbangan kelas melalui class_weight.


## **Experiment 5 - Advanced Model**

In [37]:
!pip install xgboost

In [38]:
from xgboost import XGBClassifier

### **Bobot kelas: rasio kelas tidak default terhadap default**

In [39]:
spw = (y_train == 0).sum() / (y_train == 1).sum()
print("scale_pos_weight:", round(spw, 2))

scale_pos_weight: 3.52


### **Pipeline: preprocessing + XGBoost**

In [40]:
xgb_pipe = Pipeline([
    ("prep", preprocessor),
    ("clf", XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
                          scale_pos_weight=spw, eval_metric="logloss",
                          random_state=42, n_jobs=-1)),
])

### **5-fold Cross Validation pada training set**

In [41]:
cv_f1 = cross_val_score(xgb_pipe, X_train, y_train, cv=cv, scoring="f1").mean()

### **Metrik pada test set**

In [42]:
# Latih lalu prediksi test set
xgb_pipe.fit(X_train, y_train)
y_pred = xgb_pipe.predict(X_test)

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)

print("CV F1-score    :", round(cv_f1, 4))
print("Test Accuracy  :", round(acc, 4))
print("Test Precision :", round(prec, 4))
print("Test Recall    :", round(rec, 4))
print("Test F1-score  :", round(f1, 4))

results.append({"Experiment": 5, "Model": "XGBoost",
                "CV F1": cv_f1, "Test Accuracy": acc, "Test Precision": prec,
                "Test Recall": rec, "Test F1": f1})

CV F1-score    : 0.5383
Test Accuracy  : 0.7554
Test Precision : 0.461
Test Recall    : 0.6237
Test F1-score  : 0.5301


**Perbandingan dengan model terbaik Experiment 4**

In [43]:
tabel = pd.DataFrame(results)
tabel[tabel["Model"].isin(["Random Forest (Tuned)", "XGBoost"])].round(4)

,Experiment,Model,CV F1,Test Accuracy,Test Precision,Test Recall,Test F1
5,4,Random Forest (Tuned),0.5473,0.7774,0.4973,0.5633,0.5283
6,4,Random Forest (Tuned),0.5473,0.7774,0.4973,0.5633,0.5283
7,4,Random Forest (Tuned),0.5473,0.7774,0.4973,0.5633,0.5283
8,4,Random Forest (Tuned),0.5473,0.7774,0.4973,0.5633,0.5283
9,5,XGBoost,0.5383,0.7554,0.4610,0.6237,0.5301


XGBoost pada Experiment 5 tidak memberikan performa yang lebih baik secara berarti dibandingkan Random Forest tuned dari Experiment 4. CV F1 XGBoost sebesar 0,5383 lebih rendah daripada Random Forest tuned sebesar 0,5473, sedangkan Test F1 keduanya hampir sama (0,5301 dibandingkan 0,5283). XGBoost menghasilkan Recall lebih tinggi (0,6237 dibandingkan 0,5633), tetapi Precision (0,4610 dibandingkan 0,4973) dan Accuracy (0,7554 dibandingkan 0,7774) lebih rendah. Dengan selisih F1 yang sangat kecil, kompleksitas tambahan XGBoost tidak menghasilkan peningkatan yang nyata.

In [45]:
tabel = pd.DataFrame(results).drop_duplicates(subset=["Experiment", "Model"]).reset_index(drop=True)
tabel.round(4)

,Experiment,Model,CV F1,Test Accuracy,Test Precision,Test Recall,Test F1
0,1,Baseline (Logistic Regression),0.2695,0.8079,0.6611,0.2707,0.3842
1,2,Baseline + Preprocessing,0.3587,0.8138,0.7273,0.2534,0.3758
2,3,Logistic Regression,0.3587,0.8138,0.7273,0.2534,0.3758
3,3,Decision Tree,0.4049,0.7228,0.3799,0.3997,0.3896
4,3,Random Forest,0.4720,0.8119,0.6304,0.3627,0.4605
5,4,Random Forest (Tuned),0.5473,0.7774,0.4973,0.5633,0.5283
6,5,XGBoost,0.5383,0.7554,0.4610,0.6237,0.5301
